## Topic: The Complete RAW Application With LangChain + Chroma DB (Not Completed)

In [ ]:
"""             Diagram or Process or Architecture
        ==================================================


- 1. PDF Data (local) -> extract docs 
                            -> 2. chunk docs -> ....
                                    ↓
                            -> 3. embedding chunk -> ....
                                    ↓ 
                            -> 4. vector embedding -> ....
                                    ↓ 
                            - 5.  Vector Database -> ....
                                    ↓  (retrieval response)
    - 6. user query -> embedding → LLMs  
                                    ↓ 
                            - 7.   Final response                       

"""

### 1. Import Require Library

In [ ]:
# --------------------------------------------------
# 1. Import Require Library
# --------------------------------------------------
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser


# --------------------------------------------------
# 1. Load environment variables
# --------------------------------------------------
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(
c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
python-dotenv could not parse statement starting at line 10
python-dotenv could not parse statement starting at line 20


### 2. Load the Pdf into Specific folder

In [4]:
#-------------------------------------------------------------
# STEP 2: Load the Data(PDF) - > Create an object of PyPDFLoader
# -------------------------------------------------------------
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "./data/ML-System-python-book.pdf"

# Verify file exists before loading
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"PDF file not found at: {pdf_path}")

loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"📄 Loaded {len(docs)} pages from PDF.")


📄 Loaded 326 pages from PDF.


### 3. Split the Document into smaller chunks

In [5]:
# =========================================================
# STEP 3: Split Document into Chunks
# =========================================================
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,      # 300 is too small for summaries; 1000 provides better context
    chunk_overlap=100,
)

chunks = splitter.split_documents(docs)
print(f"✂️ Total Chunks Created: {len(chunks)}")

# Helper: Combine all chunk contents into a single string for the prompt
full_text_content = "\n\n".join(chunk.page_content for chunk in chunks)

✂️ Total Chunks Created: 656


### 4. Define Prompt, embedding model  and output parser

In [ ]:
# =========================================================
# STEP 4: Create Prompt, Embedding Model, and Parser
# =========================================================
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

# Define the Prompt template
prompt_template = """Use the following context to answer the question.
If you don't know the answer based on the context, just say "I don't know". 
Do not use your prior knowledge.

Context:
{context}

Question:
{question}

Answer:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


# -----------------------------------------------------------------------------
# Define the embedding model (Download the HuggingFaceEmbeddings model)
# -----------------------------------------------------------------------------
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Output parser converts AIMessage directly to string
parser = StrOutputParser()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1336.27it/s]


### 5. Create The Chroma DB (Vector Database)

In [7]:
# =========================================================
# STEP 5: Create Chroma vector store
# =========================================================
from langchain.vectorstores import Chroma

persist_directory = "./RAG_PDF_Analysis" # dir name and location where the vector data store 

vector_store = Chroma(
    collection_name="RAG_PDF",
    embedding_function=embeddings,
    persist_directory= persist_directory
)



C:\Users\kz\AppData\Local\Temp\ipykernel_17680\1378812855.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [8]:
# for chromadb : save the database location

#  persiste the db to disk
vector_store.persist()
vector_store = None


C:\Users\kz\AppData\Local\Temp\ipykernel_17680\3388297590.py:4: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


### 6. Load the Vector Database

In [9]:
# ===========================================================================================
# STEP 6: Now we can load the persisted database from disk, and use it as normal.
# ===========================================================================================
vector_store = Chroma(
    persist_directory=persist_directory,  # dir name where the embedding vector was store locally
    embedding_function=embeddings
)


### 7. Define the retriever

In [ ]:
# ==================================
# STEP 7: Define the retriever 
#  Set the return relevant response
# ==================================

retriever = vector_store.as_retriever(search_kwargs={"k": 2})


# Define model 

In [12]:
# ------------------------------------------------------------------
# Specify model and low temperature for accurate summarization
# ------------------------------------------------------------------
llamaChatModel = ChatGroq(
    model="openai/gpt-oss-safeguard-20b",
    temperature=0.2
)

In [ ]:
# 7. Create RAG Chain (RetrievalQA)
# import chian 
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llamaChatModel,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": PROMPT},
    return_source_documents=True # # Reference Source 
)

ModuleNotFoundError: No module named 'langchain_chains'

In [21]:
## Cite sources -> optimize the response of llm

def process_llm_response(llm_response):
    print(llm_response['result'])
    print('\n\nSources:')
    for source in llm_response["source_documents"]:
        print(source.metadata['source'])

In [ ]:
# full example
query = {
    "context": "Book",
    "question": "who is the writer?"
}
llm_response = qa_chain(query)
process_llm_response(llm_response)

In [ ]:
# from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnablePassthrough


# # Define RunnableParallel to create a parallel chain
# parallel_chain = RunnableParallel({
#     'llm_response': RunnableSequence(PROMPT | llamaChatModel | parser),
#     'user_QN': RunnablePassthrough()
# })

# # full example
# query = {
#     "context": "Book",
#     "question": "who is the writer?"
# }
# llm_response = parallel_chain(query)
# process_llm_response(llm_response)